# BEAVERS: Climate Indices from EMO1 time series
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 27-04-2026<br>

**Introduction:**<br>
This notebook creates the static attributes related to the areal meteorological time series from EMO1.

In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
from tqdm.auto import tqdm

from reservoirs_lshm.utils.plots import plot_attributes
from reservoirs_lshm.utils.utils import duration_precip_indices

from ocab.config import Config

## Configuration

In [ ]:
cfg = Config('config_BEAVERS_v100.yml')

# path where the meteo time series were extracted
path_meteo = cfg.path_dataset / 'preprocessing' / 'timeseries' / 'meteo' / 'EMO1'

# save plots in folder
path_plots = cfg.path_plots / 'attributes'
path_plots.mkdir(parents=False, exist_ok=True)

print(f'Attribute tables will be saved in {cfg.path_attributes}')

## Data

### Basin outlets

In [ ]:
# load basin outlets
outlets = gpd.read_file(cfg.path_gis / 'dams.geojson').set_index('id')
print(f'{len(outlets)} basin outlets')

### Meteorology

In [ ]:
# load meteo time series
meteo = []
variables = {
    'ta_mean': 'temp', 
    'pr_mean': 'precip',
    'e0_mean': 'pet',
}
files = list(path_meteo.glob('*.parquet'))
for file in tqdm(files):
    ID = int(file.stem)

    # read time series
    df = pd.read_parquet(file).loc[ID]
    df.rename(columns=variables, inplace=True)

    # convert to xarray.Dataset
    ds = xr.Dataset.from_dataframe(df)
    ds = ds.assign_coords(id=ID).expand_dims('id')

    meteo.append(ds)

meteo = xr.concat(meteo, dim='id')

# compute precipitation as snowfall
meteo['snow'] = meteo['precip'].where(meteo['temp'] > 1, 0)

## Attributes

### Compute

In [ ]:
# average of all meteo variables
attrs = meteo.mean('time').to_pandas()
attrs.columns = [f'{col}_mean' for col in attrs.columns]
attrs.index.name = outlets.index.name

# indices
attrs['aridity'] = attrs.precip_mean / attrs.pet_mean
attrs['frac_snow'] = attrs.snow_mean / attrs.precip_mean
attrs['moisture_index'] = attrs.precip_mean - attrs.pet_mean
precip_monthly = meteo['precip'].resample({'time': 'MS'}).sum().groupby('time.month').mean()
precip_annual = meteo['precip'].resample({'time': 'YS'}).sum().mean('time')
attrs['seasonality'] = ((precip_monthly.max('month') - precip_monthly.min('month')) / precip_annual).to_pandas()

# high and dry precipitation indices
precip_extremes = {
    'high': 20, # mm
    'low': 1 # mm
}
for key, value in precip_extremes.items():
    if key == 'high':
        mask_precip = meteo['precip'] > value
    elif key == 'low':
        mask_precip = meteo['precip'] < value
    attrs[f'{key}_precip_freq'] = mask_precip.sum('time') / len(meteo.time)
    attrs[f'{key}_precip_dur'] = duration_precip_indices(mask_precip.to_pandas().transpose())

### Plot

In [ ]:
# plot attributes
r = 5
plot_attributes(
    attrs,
    outlets.geometry.x,
    outlets.geometry.y,
    ncols=4,
    extent=[-9.5, 3.5, 36, 44.5],
    save=path_plots / f'maps_climate.jpg'
)

### Export

In [ ]:
# sort index
attrs.index.name = outlets.index.name
attrs.index = attrs.index.astype(int)
attrs.sort_index(axis=0, inplace=True)

print('{0} climate indices define the climatology of {1} catchments'.format(*attrs.shape[::-1]))

# export
attrs.to_csv(cfg.path_attributes / 'climate_indices.csv', float_format='%.6f')